# Train/test split

A model's score on data it has already seen tells you little. We hide a **test set** and score on that instead. This notebook shows the split, what `random_state` does, and why one split is not the whole story.

This notebook goes with the **Train/test split** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

## 1. Load the data

`customers.csv` is a small made-up table of 80 shoppers: their **country**, **age** and **salary**, and whether they **purchased** (1) or not (0). It was generated for this course, so it is free to use. A few ages and salaries are left blank on purpose.

In [ ]:
import numpy as np
import pandas as pd

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/customers.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

## 2. Pick the features

We predict `purchased` from `age` and `salary`. Rows with a blank are dropped to keep this notebook focused on the split.

In [ ]:
data = dataset.dropna()
X = data[['age', 'salary']].values
y = data['purchased'].values
print(X.shape, 'rows, share who purchased:', y.mean().round(2))

## 3. Split

`test_size=0.25` keeps a quarter of the rows for testing. The split is random, and `random_state` is the seed: the same number always gives the same split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(len(X_train), 'train,', len(X_test), 'test')

# same seed, same split
again = train_test_split(X, y, test_size=0.25, random_state=0)
print('Same split again:', np.array_equal(again[1], X_test))

## 4. A different seed is a different split

We train the same model on five different splits. The test score moves around, because each split puts different points in the test set. A single score is a **sample**, not the truth.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

scores = []
for seed in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
    model = make_pipeline(StandardScaler(), LogisticRegression())
    model.fit(Xtr, ytr)
    scores.append(model.score(Xte, yte))
    print('random_state =', seed, '-> test accuracy', round(scores[-1], 2))

print('Average:', round(np.mean(scores), 2))

## 5. Change the size of the test set

A bigger test set gives a steadier score but leaves less data to learn from.

In [ ]:
for frac in [0.1, 0.25, 0.5]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=frac, random_state=0)
    model = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
    print(f'test_size={frac}: {len(Xtr)} train, {len(Xte)} test, test accuracy {model.score(Xte, yte):.2f}')

## Try it yourself

- Add `stratify=y` to `train_test_split`. It keeps the share of buyers the same in both sets. Does the score change?
- Run step 4 with `range(20)`. How wide is the spread of scores?